In [35]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

# path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
# grewpy.set_config('ud')
path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/SUD_French-GSD-r2.15"
grewpy.set_config('sud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value
import json 
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))


('€', 'NOUN') 45
('œuvrer', 'VERB') 12
('œuvre', 'NOUN') 115
('œuf', 'NOUN') 19
('œil', 'NOUN') 33
('île', 'NOUN') 124
('être', 'AUX') 9438
('être', 'VERB') 149
('évêque', 'NOUN') 45
('événement', 'NOUN') 50
('évènement', 'NOUN') 20
('évoquer', 'VERB') 29
('évolution', 'NOUN') 39
('évoluer', 'VERB') 65
('éviter', 'VERB') 42
('évidence', 'NOUN') 13
('éventuellement', 'ADV') 12
('éventuel', 'ADJ') 13
('évaluer', 'VERB') 11
('été', 'NOUN') 57
('étudier', 'VERB') 45
('étudiant', 'NOUN') 23
('étude', 'NOUN') 99
('étranger', 'ADJ') 45
('étranger', 'NOUN') 16
('étrange', 'ADJ') 20
('étoile', 'NOUN') 42
('étendre', 'VERB') 36
('état', 'NOUN') 149
('étape', 'NOUN') 28
('étang', 'NOUN') 11
('étage', 'NOUN') 20
('établissement', 'NOUN') 43
('établir', 'VERB') 75
('équiper', 'VERB') 22
('équipement', 'NOUN') 21
('équipe', 'NOUN') 233
('équipage', 'NOUN') 11
('épreuve', 'NOUN') 41
('épouser', 'VERB') 27
('épouse', 'NOUN') 32
('époque', 'NOUN') 100
('épisode', 'NOUN') 49
('énorme', 'ADJ') 11
('énerg

In [36]:
with open("../3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

data = { k : list() for k in match_upos }
for adv, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[adv].append(formatted_features)

unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [37]:
linear_unique_features = []
features_to_keep = ['rel_deep', 'rel_shallow']
for f in unique_features:
    if f.split(':')[3].split('=')[0] in features_to_keep:
        linear_unique_features.append(f)

linear_feature2idx = {feat : i for i, feat in enumerate(linear_unique_features)}
idx2linear_feature = {i : feat for i, feat in enumerate(linear_unique_features)}

In [38]:
X = np.zeros((len(data.keys()), len(linear_unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            if feature in linear_feature2idx:
                X[adv2idx[adv], linear_feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2978, 67)


In [39]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Fit the model for novelty detection
clf = LocalOutlierFactor(n_neighbors=2, contamination=0.1)
clf.fit(X)

# Predict outliers 
y_pred = clf.fit_predict(X)
n_outliers = np.sum(y_pred == -1)

In [40]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import pandas as pd

# Dimensionality reduction using t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_reduced = tsne.fit_transform(X)

# Prepare data for Plotly visualization
data = pd.DataFrame({
    "Component 1": X_reduced[:, 0],
    "Component 2": X_reduced[:, 1],
    "Word": unique_lemma,
    "Outlier": ["Outlier" if pred == -1 else "Inlier" for pred in y_pred],
})

# Create a scatter plot with Plotly
fig = go.Figure()

# Add scatter points
fig.add_trace(go.Scatter(
    x=data["Component 1"],
    y=data["Component 2"],
    mode='markers',
    marker=dict(size=10, color=["red" if o == "Outlier" else "blue" for o in data["Outlier"]]),
    text=data["Word"],  
    customdata=data["Outlier"],  # Inlier/Outlier status for hover
    hovertemplate="Word: %{text}<br>Status: %{customdata}<extra></extra>"
))

# Update layout
fig.update_layout(
    title="t-SNE Visualization of all lemmas with LOF Outlier Detection",
    xaxis_title="t-SNE Component 1",
    yaxis_title="t-SNE Component 2",
    width=800,
    height=600
)

# Show the plot
fig.show()

print(f"Number of outliers detected: {n_outliers}/{X.shape[0]}")

Number of outliers detected: 298/2978


In [41]:
fig.write_html("linear_configuration_outlier_detection_tsne.html")

In [42]:
outliers = []
inliers = []
for i, pred in enumerate(y_pred):
    if pred == -1:
        outliers.append(unique_lemma[i])
    else:
        inliers.append(unique_lemma[i])

inlier_pos_elements = {}
for element, pos in inliers:
    inlier_pos_elements.setdefault(pos, []).append(element)

outlier_pos_elements = {}
for element, pos in outliers:
    outlier_pos_elements.setdefault(pos, []).append(element)

In [44]:
# Prepare data for the dropdown and pie chart
pos_tags = list(set(inlier_pos_elements.keys()).union(outlier_pos_elements.keys()))
fig = go.Figure()

# Add traces for each POS
for pos in pos_tags:
    inlier_words = inlier_pos_elements.get(pos, [])
    outlier_words = outlier_pos_elements.get(pos, [])
    
    # Calculate percentages
    total = len(inlier_words) + len(outlier_words)
    inlier_percentage = len(inlier_words) / total * 100 if total > 0 else 0
    outlier_percentage = len(outlier_words) / total * 100 if total > 0 else 0
    
    # Create hover text
    inlier_hover_text = '<br>'.join(inlier_words)
    outlier_hover_text = '<br>'.join(outlier_words)
    
    # Add a trace for this POS
    fig.add_trace(go.Pie(
        labels=['Inliers', 'Outliers'],
        values=[len(inlier_words), len(outlier_words)],
        hovertext=[inlier_hover_text, outlier_hover_text],
        hoverinfo="text",
        name=pos,
        marker=dict(colors=['blue', 'red'])
    ))

# Update layout with dropdown
fig.update_layout(
    title="POS Distribution of Inliers and Outliers - Linear Configuration Features only",
    updatemenus=[
        {
            "buttons": [
                {
                    "label": pos,
                    "method": "update",
                    "args": [
                        {"visible": [i == idx for i in range(len(pos_tags))]},
                        {"title": f"POS Distribution for {pos}"}
                    ]
                }
                for idx, pos in enumerate(pos_tags)
            ],
            "direction": "down",
            "showactive": True
        }
    ]
)

# Set all traces except the first one to be initially hidden
for i in range(1, len(pos_tags)):
    fig.data[i].visible = False

# Show the chart
fig.show()

In [45]:
fig.write_html("linear_configuration_outlier_detection_pos_distribution.html")

In [46]:
negative_outlier_factors = clf.negative_outlier_factor_
outlier_scores = {unique_lemma[i]: negative_outlier_factors[i] for i in range(len(unique_lemma))}

score_pos_elements = {}
for element, pos in unique_lemma:
    score_pos_elements.setdefault(pos, []).append((element, outlier_scores[(element, pos)]))

In [49]:
# Convert dictionary to a structured list
data = []
for pos, words in score_pos_elements.items():
    for word, score in words:
        log_score = np.log1p(abs(score))
        status = 'Outlier' if word in outlier_pos_elements.get(pos, []) else 'Inlier'
        color = 'red' if status == 'Outlier' else 'blue'
        data.append({'POS': pos, 'Word': word, 'NOF': log_score, 'Status': status, 'Color': color})

df = pd.DataFrame(data)

# Prepare dropdown and scatter plot
pos_tags = list(score_pos_elements.keys())
fig = go.Figure()

# Add traces for each POS
def create_trace(pos):
    filtered_df = df[df['POS'] == pos]
    return go.Scatter(
        x=np.arange(len(filtered_df)),
        y=filtered_df['NOF'],
        text=[f"{row['Word']} ({row['Status']}), NOF: {row['NOF']:.2f}" for _, row in filtered_df.iterrows()],
        mode='markers',
        marker=dict(color=filtered_df['Color'], size=10),
        name=pos,
        hoverinfo='text'
    )

for pos in pos_tags:
    fig.add_trace(create_trace(pos))

# Update layout with dropdown
fig.update_layout(
    title="Negative Outlier Factor per POS - Linear Configuration Features only",
    xaxis=dict(showticklabels=False),
    yaxis=dict(type='log', title="Log Scale of NOF"),
    updatemenus=[
        {
            "buttons": [
                {
                    "label": pos,
                    "method": "update",
                    "args": [
                        {"visible": [i == idx for i in range(len(pos_tags))]},
                        {"title": f"Negative Outlier Factor for {pos} - Linear Configuration Features only"}
                    ]
                }
                for idx, pos in enumerate(pos_tags)
            ],
            "direction": "down",
            "showactive": True
        }
    ]
)

# Set all traces except the first one to be initially hidden
for i in range(1, len(pos_tags)):
    fig.data[i].visible = False

# Show the chart
fig.show()

In [50]:
fig.write_html("linear_configuration_outlier_detection_nof_per_pos.html")

In [30]:
df

,POS,Word,NOF,Status,Color
0,NOUN,$,0.763736,Inlier,blue
1,NOUN,%,0.728039,Inlier,blue
2,NOUN,C,15.037309,Outlier,red
3,NOUN,CD,0.801987,Inlier,blue
4,NOUN,DVD,0.789666,Inlier,blue
...,...,...,...,...,...
2973,SCONJ,que,0.904190,Inlier,blue
2974,SCONJ,si,0.782974,Inlier,blue
2975,X,del,0.649782,Inlier,blue
2976,X,etc.,1.665908,Outlier,red


In [51]:
import plotly.express as px
# Sort the DataFrame by Score
data = []
for pos, words in score_pos_elements.items():
    for word, score in words:
        log_score = np.log1p(abs(score))
        status = 'Outlier' if word in outlier_pos_elements.get(pos, []) else 'Inlier'
        color = 'red' if status == 'Outlier' else 'blue'
        data.append({'POS': pos, 'Word': word, 'NOF': log_score, 'Status': status, 'Color': color})

df = pd.DataFrame(data)

df = df.sort_values(by="NOF")

# Add a color column based on the Outlier status
df['Color'] = df['Status'].apply(lambda x: 'blue' if x == 'Inlier' else 'red')

# Create the scatter plot
fig = px.scatter(
    df,
    x="NOF",
    y="POS",
    color="Color",  # Use the color column for coloring
    hover_data={"Color": False, "NOF": True, "Word": True},  # Show Complete_feature on hover
    color_discrete_map={"blue": "blue", "red": "red"}  # Map colors explicitly
)

# Update layout for better visualization
fig.update_traces(marker=dict(size=10))  # Adjust marker size
fig.update_layout(
    title="Local outlier factor clustering by POS - linear configuration features only",
    xaxis_title="NOF",
    yaxis_title="POS",
    height=800,  # Set the height of the plot
    showlegend=False  # Hide legend since colors are self-explanatory
)

# Show the plot
fig.show()

In [52]:
fig.write_html("linear_configuration_outlier_detection_nof_per_pos_axis.html")